# Predictive Model for POL via Spectral Signature Matching

## Method Overview

This notebook implements a **POL predictive model** (sucrose content in sugarcane)
based on the concept of **group spectral signatures**.

### What is a Group Spectral Signature?

A *spectral signature* is the **average profile** of a set of climatic, humidity,
and vegetation index (NDVI) variables that characterizes a specific group of fields.
Each group is defined by the unique combination of:

- `PAIS` (Country)
- `CUADRANTE` (Quadrant / Zone)
- `GRUPO_VARIEDAD` (Variety Group)
- `tercio` (Growth Third)
- `MADURACION_PRODUCTO` (Ripening Product)

### Prediction Mechanism

Given a **new field** for which POL is unknown, we compare its variables
against the spectral signature of each group using the **sum of squared differences**
(squared Euclidean distance):

$$D_g = \sum_{i=1}^{n} (x_i - \mu_{g,i})^2$$

Where:
- $x_i$ = value of variable $i$ for the new field
- $\mu_{g,i}$ = mean of variable $i$ within group $g$
- $n$ = number of predictor variables (20 variables)

The winning group $g^*$ minimizes $D_g$:

$$g^* = \arg\min_g \sum_{i=1}^{n} (x_i - \mu_{g,i})^2$$

The **predicted POL** is the mean POL of that group:

$$\hat{POL} = \mu_{g^*, \text{POL}}$$

### Predictor Variables (20 variables)

| Category | Variables |
|----------|-----------|
| Precipitation | `PREC15`, `PREC30`, `PREC60` |
| Mean Temperature | `TEMP15` |
| Minimum Temperature | `TEMP_MIN15`, `TEMP_MIN30`, `TEMP_MIN60` |
| Solar Radiation | `RAD15`, `RAD30`, `RAD60` |
| Humidity | `HUMEDAD_POND`, `HUMEDAD_POND15`, `HUMEDAD_POND30`, `HUMEDAD_POND60`, `HUMEDAD_EN_ELONGACION_2` |
| NDVI | `NDVI_POND`, `NDVI_POND15`, `NDVI_POND30`, `NDVI_POND60`, `NDVI_EN_ELONGACION_2` |

## Step 1: Library Imports

In [2]:
# ==============================================================
# STEP 1: Import required libraries
# ==============================================================

import pandas as pd        # Tabular data manipulation
import numpy as np         # Numerical and vectorized operations
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")
print(f"  pandas  version: {pd.__version__}")
print(f"  numpy   version: {np.__version__}")

Libraries imported successfully.
  pandas  version: 2.2.3
  numpy   version: 2.2.4


## Step 2: Variable Definition

Before loading the data, we define the columns assigned to each role:

- **Grouping columns**: define the group / spectral signature identity
- **Target variable**: `POL` (the variable to predict)
- **Predictor variables**: the 20 climatic, humidity, and NDVI variables

In [3]:
# ==============================================================
# STEP 2: Column role definitions
# ==============================================================

# --- Columns that define each group (spectral signature identity) ---
COLS_GROUP = [
    'PAIS',
    'CUADRANTE',
    'GRUPO_VARIEDAD',
    'tercio',
    'MADURACION_PRODUCTO'
]

# --- Target variable (the one we aim to predict) ---
COL_TARGET = 'POL'

# --- Predictor variables (the 20 spectral signature variables) ---
COLS_PREDICTORS = [
    # Accumulated precipitation over 15, 30, and 60-day windows
    'PREC15', 'PREC30', 'PREC60',
    # Mean and minimum temperature over time windows
    'TEMP15', 'TEMP_MIN15', 'TEMP_MIN30', 'TEMP_MIN60',
    # Solar radiation over time windows
    'RAD15', 'RAD30', 'RAD60',
    # Weighted humidity across windows and elongation stage
    'HUMEDAD_POND', 'HUMEDAD_POND15', 'HUMEDAD_POND30',
    'HUMEDAD_POND60', 'HUMEDAD_EN_ELONGACION_2',
    # Normalized Difference Vegetation Index (NDVI)
    'NDVI_POND', 'NDVI_POND15', 'NDVI_POND30',
    'NDVI_POND60', 'NDVI_EN_ELONGACION_2'
]

print("Variable roles defined:")
print(f"  Grouping columns    : {len(COLS_GROUP)}")
print(f"  Target variable     : {COL_TARGET}")
print(f"  Predictor variables : {len(COLS_PREDICTORS)}")

Variable roles defined:
  Grouping columns    : 5
  Target variable     : POL
  Predictor variables : 20


## Step 3: Loading the CSV File

We load the original dataset, which contains sugarcane field records with
climatic variables, vegetation indices, and quality measurements (POL, BRIX, etc.).

> **Note**: Adjust the `FILE_PATH` variable to the location of your
> `FirmaEspectral.csv` file.

In [4]:
# ==============================================================
# STEP 3: Load the CSV file
# ==============================================================

# Adjust this path to where the file is located on your system
FILE_PATH = "FirmaEspectral.csv"

# Load the full dataset
df = pd.read_csv(FILE_PATH)

print("File loaded successfully.")
print(f"  Rows    : {df.shape[0]:,}")
print(f"  Columns : {df.shape[1]:,}")
print()

# Verify that all required columns are present
all_required_cols = COLS_GROUP + [COL_TARGET] + COLS_PREDICTORS
missing_cols = [c for c in all_required_cols if c not in df.columns]

if missing_cols:
    print(f"WARNING: {len(missing_cols)} required column(s) not found in the file:")
    for c in missing_cols:
        print(f"  [MISSING] {c}")
else:
    print("All required columns are present in the dataset.")

# Preview of relevant columns
print()
print("Preview (first 3 rows, key columns):")
df[COLS_GROUP + [COL_TARGET] + COLS_PREDICTORS[:5]].head(3)

File loaded successfully.
  Rows    : 8,971
  Columns : 108

All required columns are present in the dataset.

Preview (first 3 rows, key columns):


,PAIS,CUADRANTE,GRUPO_VARIEDAD,tercio,MADURACION_PRODUCTO,POL,PREC15,PREC30,PREC60,TEMP15,TEMP_MIN15
0,GT01,CENTRO ALTO,CG98-78,segundo,1,15.19782,0.0,0.0,0.0,NaN,NaN
1,GT01,CENTRO ALTO,CG98-78,segundo,1,15.74950,14.5,6.4,105.4,20.56667,13.80000
2,GT01,CENTRO ALTO,CG98-46,segundo,1,14.92181,6.7,10.4,223.5,20.68000,13.46667


## Step 4: Data Quality Assessment

Before computing spectral signatures, we inspect the quality of the dataset:
the extent of missing values across predictor columns and the target variable.

Missing values are not a critical issue for this method because `pandas` automatically
excludes `NaN` entries when computing group means (`groupby + mean`).
However, their magnitude should be documented for transparency.

In [5]:
# ==============================================================
# STEP 4: Missing value assessment
# ==============================================================

analysis_cols = COLS_GROUP + [COL_TARGET] + COLS_PREDICTORS

print("=" * 65)
print("MISSING VALUE SUMMARY FOR RELEVANT COLUMNS")
print("=" * 65)

null_summary = []
for col in analysis_cols:
    n_missing = df[col].isna().sum()
    pct = 100 * n_missing / len(df)
    null_summary.append({
        'Column'     : col,
        'Missing'    : n_missing,
        'Percentage' : f"{pct:.1f}%",
        'Flag'       : "WARNING" if pct > 3 else ("OK" if n_missing == 0 else "MINOR")
    })

df_nulls = pd.DataFrame(null_summary)
print(df_nulls.to_string(index=False))

print()
print("Target variable (POL) — descriptive statistics:")
print(df[COL_TARGET].describe().round(4))

MISSING VALUE SUMMARY FOR RELEVANT COLUMNS
                 Column  Missing Percentage    Flag
                   PAIS        0       0.0%      OK
              CUADRANTE        0       0.0%      OK
         GRUPO_VARIEDAD        0       0.0%      OK
                 tercio        0       0.0%      OK
    MADURACION_PRODUCTO        0       0.0%      OK
                    POL        0       0.0%      OK
                 PREC15        0       0.0%      OK
                 PREC30        0       0.0%      OK
                 PREC60        0       0.0%      OK
                 TEMP15       41       0.5%   MINOR
             TEMP_MIN15       41       0.5%   MINOR
             TEMP_MIN30       41       0.5%   MINOR
             TEMP_MIN60       41       0.5%   MINOR
                  RAD15       41       0.5%   MINOR
                  RAD30       41       0.5%   MINOR
                  RAD60       41       0.5%   MINOR
           HUMEDAD_POND       95       1.1%   MINOR
         HUMEDAD_POND

## Step 5: Computing Spectral Signatures

This is the central step of the model.

For each unique combination of
(`PAIS`, `CUADRANTE`, `GRUPO_VARIEDAD`, `tercio`, `MADURACION_PRODUCTO`),
we compute the **mean** of all predictor variables and the target variable POL.

The result is a table where **each row is a spectral signature**,
i.e., the average climatic-vegetative profile of that group.

Formally, for group $g$:

$$\mu_{g,j} = \frac{1}{N_g} \sum_{k=1}^{N_g} x_{k,j}$$

Where $N_g$ is the number of records in group $g$, and $x_{k,j}$ is the value
of variable $j$ for record $k$.

In [6]:
# ==============================================================
# STEP 5: Compute spectral signatures (group-level means)
# ==============================================================

# Columns to average: predictors + target
cols_to_average = COLS_PREDICTORS + [COL_TARGET]

# Group by the 5 categorical columns and compute column-wise means.
# NaN values are automatically excluded from the mean computation.
signatures = (
    df
    .groupby(COLS_GROUP)[cols_to_average]
    .mean()
    .reset_index()
)

# Append group size (number of records per group)
group_sizes = df.groupby(COLS_GROUP).size().reset_index(name='N_RECORDS')
signatures = signatures.merge(group_sizes, on=COLS_GROUP)

print("Spectral signatures computed successfully.")
print(f"  Total groups (signatures): {len(signatures):,}")
print()
print("Preview of spectral signatures (first 3 rows):")
signatures.head(3)

Spectral signatures computed successfully.
  Total groups (signatures): 620

Preview of spectral signatures (first 3 rows):


,PAIS,CUADRANTE,GRUPO_VARIEDAD,tercio,MADURACION_PRODUCTO,PREC15,PREC30,PREC60,TEMP15,TEMP_MIN15,...,HUMEDAD_POND30,HUMEDAD_POND60,HUMEDAD_EN_ELONGACION_2,NDVI_POND,NDVI_POND15,NDVI_POND30,NDVI_POND60,NDVI_EN_ELONGACION_2,POL,N_RECORDS
0,GT01,CENTRO ALTO,CG,primero,1,28.55,59.9,260.1,21.536665,15.463330,...,70.614165,71.049500,74.122340,0.6700,0.66667,0.669165,0.683500,0.708925,15.007180,2
1,GT01,CENTRO ALTO,CG,segundo,0,2.40,0.9,26.1,22.295000,16.033335,...,71.568335,72.407085,73.228815,0.6225,0.61750,0.733335,0.730832,0.745120,15.359368,4
2,GT01,CENTRO ALTO,CG,segundo,1,19.10,10.9,196.6,20.140000,15.333330,...,69.693330,70.781670,72.487500,0.6000,0.62000,0.613330,0.728330,0.755500,15.807450,1


## Step 6: Saving the Spectral Signatures

We save the spectral signatures to a CSV file for later use,
avoiding the need to recompute them from the raw dataset.

This step is analogous to the **model training and serialization** phase
in a standard machine learning pipeline.

In [7]:
# ==============================================================
# STEP 6: Save spectral signatures to disk
# ==============================================================

OUTPUT_PATH = "spectral_signatures.csv"
signatures.to_csv(OUTPUT_PATH, index=False)

print(f"Spectral signatures saved to: '{OUTPUT_PATH}'")
print(f"  File dimensions: {len(signatures)} rows x {len(signatures.columns)} columns")
print()
print("Columns in the output file:")
for i, col in enumerate(signatures.columns, 1):
    print(f"  {i:3d}. {col}")

Spectral signatures saved to: 'spectral_signatures.csv'
  File dimensions: 620 rows x 27 columns

Columns in the output file:
    1. PAIS
    2. CUADRANTE
    3. GRUPO_VARIEDAD
    4. tercio
    5. MADURACION_PRODUCTO
    6. PREC15
    7. PREC30
    8. PREC60
    9. TEMP15
   10. TEMP_MIN15
   11. TEMP_MIN30
   12. TEMP_MIN60
   13. RAD15
   14. RAD30
   15. RAD60
   16. HUMEDAD_POND
   17. HUMEDAD_POND15
   18. HUMEDAD_POND30
   19. HUMEDAD_POND60
   20. HUMEDAD_EN_ELONGACION_2
   21. NDVI_POND
   22. NDVI_POND15
   23. NDVI_POND30
   24. NDVI_POND60
   25. NDVI_EN_ELONGACION_2
   26. POL
   27. N_RECORDS


## Step 7: Prediction Function

We now define the **prediction function**. Given a new field with known values
for the 20 predictor variables (but unknown POL), the function:

1. **Computes the squared distance** between the new field and each spectral signature:

$$D_g = \sum_{i=1}^{20} (x_i - \mu_{g,i})^2$$

2. **Identifies the nearest group** (minimum $D_g$)
3. **Returns** the mean POL of that group, along with group identification details

### Note on Normalization

The predictor variables operate on very different scales
(e.g., `PREC60` may reach hundreds of mm, while `NDVI_POND` ranges from 0 to 1).
Without normalization, high-magnitude variables would dominate the distance metric.
The function offers **Z-score standardization** prior to distance computation,
which is the statistically recommended approach.

$$z_i = \frac{x_i - \bar{\mu}_i}{s_i}$$

Where $\bar{\mu}_i$ and $s_i$ are the mean and standard deviation of variable $i$
computed across all spectral signatures.

In [8]:
# ==============================================================
# STEP 7: Prediction function
# ==============================================================

def predict_pol(new_field: dict,
                signatures: pd.DataFrame,
                predictor_cols: list,
                target_col: str = 'POL',
                group_cols: list = None,
                normalize: bool = True,
                top_n: int = 3) -> dict:
    """
    Predicts the POL value of a new field by comparing its spectral signature
    against all group signatures using minimum squared Euclidean distance.

    Parameters
    ----------
    new_field       : dict mapping predictor column names to numeric values.
                      Variables absent from the dict, set to None, or NaN
                      are automatically excluded from the distance computation.
    signatures      : DataFrame of group spectral signatures (output of Step 5).
    predictor_cols  : list of column names to use for distance computation.
    target_col      : name of the target column (default: 'POL').
    group_cols      : list of columns identifying each group (used in the report).
    normalize       : if True, applies Z-score standardization before computing
                      distances (recommended when variable scales differ).
    top_n           : number of nearest groups to include in the output report.

    Returns
    -------
    dict containing:
        'predicted_pol'      : predicted POL value (float)
        'group'              : dict identifying the winning group
        'distance'           : squared distance to the winning group
        'n_records'          : number of records composing the winning group
        'top_groups'         : DataFrame of the top_n nearest groups
        'variables_used'     : list of variables included in the computation
        'variables_missing'  : list of variables excluded (absent or NaN)
        'normalized'         : whether normalization was applied
    """
    if group_cols is None:
        group_cols = COLS_GROUP

    # ------------------------------------------------------------------
    # 1. Identify which predictor variables are available and valid
    # ------------------------------------------------------------------
    variables_used = [
        c for c in predictor_cols
        if c in new_field
        and new_field[c] is not None
        and not np.isnan(float(new_field[c]))
    ]
    variables_missing = [c for c in predictor_cols if c not in variables_used]

    if not variables_used:
        raise ValueError(
            "The new field contains no valid predictor variables. "
            "Ensure at least one variable is non-null."
        )

    if variables_missing:
        print(
            f"Note: {len(variables_missing)} variable(s) excluded from computation "
            f"(absent or NaN): {variables_missing}"
        )

    # ------------------------------------------------------------------
    # 2. Extract the new field values as a numpy vector
    # ------------------------------------------------------------------
    vector_new = np.array([float(new_field[c]) for c in variables_used])

    # ------------------------------------------------------------------
    # 3. Extract the signature matrix (available variables only)
    # ------------------------------------------------------------------
    sig_sub = signatures[variables_used].copy()

    # Impute NaN in signatures with column mean (conservative fallback)
    for col in variables_used:
        if sig_sub[col].isna().any():
            sig_sub[col] = sig_sub[col].fillna(sig_sub[col].mean())

    sig_matrix = sig_sub.values   # shape: (n_groups, n_variables)

    # ------------------------------------------------------------------
    # 4. Optional Z-score normalization
    #    Parameters are computed from the signature matrix, not raw data,
    #    so normalization is relative to the distribution of group profiles.
    # ------------------------------------------------------------------
    if normalize:
        col_means = sig_matrix.mean(axis=0)
        col_stds  = sig_matrix.std(axis=0)
        col_stds  = np.where(col_stds == 0, 1, col_stds)  # prevent division by zero

        sig_matrix_norm = (sig_matrix - col_means) / col_stds
        vector_new_norm = (vector_new  - col_means) / col_stds
    else:
        sig_matrix_norm = sig_matrix
        vector_new_norm = vector_new

    # ------------------------------------------------------------------
    # 5. Compute squared Euclidean distance for every group
    #    D_g = sum_i (x_i - mu_g_i)^2
    # ------------------------------------------------------------------
    differences = sig_matrix_norm - vector_new_norm   # broadcasting: (n_groups, n_vars)
    distances   = (differences ** 2).sum(axis=1)      # row-wise sum: (n_groups,)

    # ------------------------------------------------------------------
    # 6. Identify the winning group (minimum distance)
    # ------------------------------------------------------------------
    winner_idx = int(np.argmin(distances))

    # Build top-N report
    top_indices = np.argsort(distances)[:top_n]
    top_groups  = signatures.iloc[top_indices][group_cols + [target_col, 'N_RECORDS']].copy()
    top_groups.insert(0, 'RANK', range(1, len(top_indices) + 1))
    top_groups['DISTANCE'] = distances[top_indices].round(6)
    top_groups = top_groups.reset_index(drop=True)

    # ------------------------------------------------------------------
    # 7. Assemble result dictionary
    # ------------------------------------------------------------------
    winner_row = signatures.iloc[winner_idx]
    group_info = {col: winner_row[col] for col in group_cols}

    result = {
        'predicted_pol'     : round(float(winner_row[target_col]), 6),
        'group'             : group_info,
        'distance'          : round(float(distances[winner_idx]), 6),
        'n_records'         : int(winner_row['N_RECORDS']),
        'top_groups'        : top_groups,
        'variables_used'    : variables_used,
        'variables_missing' : variables_missing,
        'normalized'        : normalize
    }

    return result


print("Function 'predict_pol' defined successfully.")

Function 'predict_pol' defined successfully.


## Step 8: Result Reporting Function

An auxiliary function to display the prediction result in a structured,
readable format.

In [9]:
# ==============================================================
# STEP 8: Result reporting function
# ==============================================================

def print_result(result: dict):
    """
    Prints a structured report of the output from predict_pol().
    """
    sep = "=" * 65

    print(sep)
    print("  POL PREDICTION REPORT")
    print(sep)

    norm_label = "normalized" if result['normalized'] else "not normalized"
    print(f"\n  Predicted POL  : {result['predicted_pol']:.4f}")
    print(f"  Distance       : {result['distance']:.6f}  ({norm_label})")
    print(f"  Group size     : {result['n_records']} records")

    print(f"\n  NEAREST GROUP (winning spectral signature):")
    for key, value in result['group'].items():
        print(f"    {key:<28} : {value}")

    print(f"\n  TOP {len(result['top_groups'])} NEAREST GROUPS:")
    print()
    print(result['top_groups'].to_string(index=False))

    if result['variables_missing']:
        print(f"\n  Variables excluded from computation "
              f"({len(result['variables_missing'])} total):")
        for v in result['variables_missing']:
            print(f"    - {v}")
    else:
        print(f"\n  All {len(result['variables_used'])} predictor variables "
              f"were used in the computation.")

    print()
    print(sep)


print("Function 'print_result' defined successfully.")

Function 'print_result' defined successfully.


## Step 9: Prediction Example

We test the model with a concrete example.

The values below correspond to a field with known climatic and vegetation
measurements for which we wish to predict POL.

> **To use with your own field data**: replace the numeric values in the
> `new_field_example` dictionary with your actual field measurements.
> Variables that are unavailable can be omitted or set to `None` / `np.nan`;
> the function will exclude them from the distance computation.

In [10]:
# ==============================================================
# STEP 9: Prediction example with a new field
# ==============================================================

# Define the new field's predictor variable values.
# Replace these values with those of your actual field.
new_field_example = {
    # --- Accumulated precipitation (mm) ---
    'PREC15'                 : 23.80,
    'PREC30'                 : 8.00,
    'PREC60'                 : 87.60,

    # --- Mean temperature (degrees C) ---
    'TEMP15'                 : 24.33,

    # --- Minimum temperature (degrees C) ---
    'TEMP_MIN15'             : 17.48,
    'TEMP_MIN30'             : 17.00,
    'TEMP_MIN60'             : 18.69,

    # --- Solar radiation (MJ/m2) ---
    'RAD15'                  : 0.1103,
    'RAD30'                  : 0.1286,
    'RAD60'                  : 0.0929,

    # --- Humidity (%) ---
    'HUMEDAD_POND'           : 71.22,
    'HUMEDAD_POND15'         : 71.99,
    'HUMEDAD_POND30'         : 71.56,
    'HUMEDAD_POND60'         : 72.70,
    'HUMEDAD_EN_ELONGACION_2': 74.36,

    # --- Normalized Difference Vegetation Index (0 to 1) ---
    'NDVI_POND'              : 0.700,
    'NDVI_POND15'            : 0.710,
    'NDVI_POND30'            : 0.743,
    'NDVI_POND60'            : 0.790,
    'NDVI_EN_ELONGACION_2'   : 0.761,
}

# Run the prediction
result = predict_pol(
    new_field        = new_field_example,
    signatures       = signatures,
    predictor_cols   = COLS_PREDICTORS,
    target_col       = COL_TARGET,
    group_cols       = COLS_GROUP,
    normalize        = True,   # Recommended: standardize before distance computation
    top_n            = 5       # Report the 5 nearest groups
)

# Display the result
print_result(result)

  POL PREDICTION REPORT

  Predicted POL  : 14.4593
  Distance       : 0.308997  (normalized)
  Group size     : 4 records

  NEAREST GROUP (winning spectral signature):
    PAIS                         : GT01
    CUADRANTE                    : CENTRO OESTE MEDIO
    GRUPO_VARIEDAD               : SP
    tercio                       : segundo
    MADURACION_PRODUCTO          : 1

  TOP 5 NEAREST GROUPS:

 RANK PAIS          CUADRANTE GRUPO_VARIEDAD  tercio  MADURACION_PRODUCTO       POL  N_RECORDS  DISTANCE
    1 GT01 CENTRO OESTE MEDIO             SP segundo                    1 14.459285          4  0.308997
    2 GT01  CENTRO OESTE BAJO             ME segundo                    1 15.248200          1  3.340855
    3 GT01 CENTRO OESTE MEDIO        CG98-78 segundo                    1 14.383280          2  3.957485
    4 GT01  CENTRO OESTE BAJO       CG02-163 segundo                    0 15.388661         13  4.228378
    5 GT01  CENTRO OESTE BAJO      CP72-2086 segundo               

## Step 10: Exploratory Analysis of Spectral Signatures

We examine the distribution of mean POL across all groups, and identify
which groups exhibit the highest and lowest average sucrose content.
This provides context for interpreting prediction outputs.

In [11]:
# ==============================================================
# STEP 10: Exploratory analysis of spectral signatures
# ==============================================================

print("=" * 65)
print("POL DISTRIBUTION ACROSS GROUPS")
print("=" * 65)
print()
print(f"  Total spectral signatures (groups): {len(signatures)}")
print()
print("  Descriptive statistics of mean POL per group:")
print(signatures[COL_TARGET].describe().round(4).to_string())

print()
print("=" * 65)
print("TOP 10 GROUPS BY MEAN POL (highest sucrose content)")
print("=" * 65)
top_pol = (
    signatures[COLS_GROUP + [COL_TARGET, 'N_RECORDS']]
    .sort_values(COL_TARGET, ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top_pol.index += 1
print(top_pol.to_string())

print()
print("=" * 65)
print("BOTTOM 10 GROUPS BY MEAN POL (lowest sucrose content)")
print("=" * 65)
bot_pol = (
    signatures[COLS_GROUP + [COL_TARGET, 'N_RECORDS']]
    .sort_values(COL_TARGET, ascending=True)
    .head(10)
    .reset_index(drop=True)
)
bot_pol.index += 1
print(bot_pol.to_string())

POL DISTRIBUTION ACROSS GROUPS

  Total spectral signatures (groups): 620

  Descriptive statistics of mean POL per group:
count    620.0000
mean      15.4795
std        1.1126
min       11.2610
25%       14.9080
50%       15.4950
75%       16.0807
max       19.2566

TOP 10 GROUPS BY MEAN POL (highest sucrose content)
    PAIS           CUADRANTE GRUPO_VARIEDAD   tercio  MADURACION_PRODUCTO        POL  N_RECORDS
1   GT01    CENTRO ESTE ALTO     CG04-10267  tercero                    1  19.256570          1
2   GT01   CENTRO ESTE MEDIO       CG02-163  tercero                    1  18.710000          1
3   GT01    CENTRO ESTE ALTO       RB845210  tercero                    1  18.642860          1
4   GT01  CENTRO OESTE MEDIO      CP88-1165  tercero                    1  18.565850          1
5   GT01    CENTRO ESTE ALTO       CG03-025  tercero                    1  18.549030          2
6   GT01         CENTRO ALTO             CG  tercero                    0  18.546000          1
7   GT01

## Summary: Complete Pipeline
```python
# COMPLETE PIPELINE IN 4 STEPS:

# 1. LOAD raw data
df = pd.read_csv("FirmaEspectral.csv")

# 2. COMPUTE spectral signatures (performed once)
signatures = df.groupby(COLS_GROUP)[COLS_PREDICTORS + ['POL']].mean().reset_index()
group_sizes = df.groupby(COLS_GROUP).size().reset_index(name='N_RECORDS')
signatures = signatures.merge(group_sizes, on=COLS_GROUP)

# 3. SAVE signatures for future use
signatures.to_csv("spectral_signatures.csv", index=False)

# 4. PREDICT POL for a new field
result = predict_pol(new_field, signatures, COLS_PREDICTORS)
print_result(result)
```

### Method Properties

| Aspect | Description |
|--------|-------------|
| **Interpretability** | Output includes the matched group and its distance, enabling audit |
| **Robustness** | Aggregation to group means reduces the influence of individual noisy records |
| **Missing data tolerance** | Variables absent from the new field are excluded from the distance computation |
| **Computational efficiency** | Prediction is near-instantaneous via vectorized array operations |

### Known Limitations

- If a new field is substantially different from all known groups (out-of-distribution),
  the prediction constitutes extrapolation and should be treated with caution.
  The magnitude of the minimum distance is a practical diagnostic.
- Predictive quality depends on the representativeness of the training dataset
  relative to the target population of fields.
- Z-score normalization is computed relative to the signature matrix
  (group centroids), not the raw dataset. This is appropriate for the
  comparison context but should be noted when comparing scale statistics.
- The method does not quantify prediction uncertainty.
  A confidence interval based on within-group POL variance is a
  recommended extension.